In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['TRANSFORMERS_CACHE'] = '/mount/arbeitsdaten/asr-2/vaethdk/resources/weights/llm'

In [2]:
from tqdm.auto import tqdm
from typing import List, Tuple
DEVICE = 'cuda:0'
import torch

/home/users2/vaethdk/.virtualenvs/cts_al/lib64/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
torch.cuda.device_count()

1

In [4]:
import sys
sys.path.append('../../..')
print(os.path.realpath("."))

/mount/arbeitsdaten41/projekte/asr-2/vaethdk/cts_activelearning/conversational-tree-search/generation/reimburse/llama3


In [5]:
import transformers
from transformers import AutoModelForCausalLM, pipeline, AutoTokenizer, set_seed
from data.dataset import ReimburseGraphDataset, DataAugmentationLevel, NodeType, DialogNode, Question

/home/users2/vaethdk/.virtualenvs/cts_al/lib64/python3.10/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [6]:
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [7]:
pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.float16},
    device='cuda:0'
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.03it/s]


In [8]:
import re

def unique_lines(generated_text: str) -> List[str]:
    # NOTE: text should be formatted as numbered list
    unique = []
    unique_lower = set()
    for idx, line in enumerate(generated_text.split("<br>")):
        line_cleaned = line.strip()
        if not line_cleaned.lower() in unique_lower:
            unique.append(line_cleaned)
            unique_lower.add(line_cleaned.lower())

    return unique

## Generate Answer Synonyms

In [9]:
from data.dataset import ReimburseGraphDataset

In [10]:
from typing import List

def generate_prompt(system: str, user: str):
    return [
        {
            "role": "system",
            "content": system
        },
        {
            "role": "user",
            "content": user
        }
    ]

import re

def _split_numbered_list(input_string):
    # Regular expression to match the numbered items
    pattern = re.compile(r'(\d+\.\s+)(.*?)((?=\d+\.\s)|$)', re.DOTALL)
    matches = pattern.findall(input_string)
    
    # Extract the text for each item
    items = [match[1].strip() for match in matches]
    
    return items


def generate_output(prompt: List[str], temperature: float = 0.7, max_new_tokens: int = 7500) -> List[str]:
    outputs = pipeline(
        prompt,
        max_new_tokens=max_new_tokens,
        # do_sample = False
        temperature=temperature
    )
    generated_text: str = outputs[0]["generated_text"][-1]['content']

    # split outputs
    outputs = generated_text.split("<br>")
    if len(outputs) == 1: 
        # maybe the list was formatted as numbers
        print("Trying to split by number...")
        outputs = _split_numbered_list(generated_text)

    uniques = set()
    results = []
    duplicates = 0
  
    # collect outputs, and trim
    for result in outputs:
        cleaned = result.strip().strip("\n")
        if len(cleaned.split(" ")) > 25:
            number_split = _split_numbered_list(cleaned)
            if len(number_split) > 1:
                print("LONG, but number splittable")
                for number_line in number_split:
                    cleaned = number_line.strip().strip("\n")
                    cleaned = re.sub(r'^\d+\s*(\.?\s*)', '', cleaned)
                    cleaned = cleaned.replace('<response>', "").replace('</response>', '')
                    if len(cleaned.split(" ")) > 25:
                        print("LONG", cleaned)
                    else:
                        if not cleaned.lower() in uniques:
                            results.append(cleaned.strip())
                            uniques.add(cleaned.lower())
            else:
                print("LONG:", cleaned)
        else:
            cleaned = re.sub(r'^\d+\s*(\.?\s*)', '', cleaned)
            cleaned = cleaned.replace('<response>', "").replace('</response>', '')
            if not cleaned.lower() in uniques:
                results.append(cleaned.strip())
                uniques.add(cleaned.lower())
    print(" - duplicates:", duplicates)
    return results

In [11]:
human_data_train = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', False, augmentation=DataAugmentationLevel.NONE, resource_dir="../../../resources")

- not using synonyms
===== Dataset Statistics =====
- files:  en/reimburse/train_graph.json en/reimburse/train_answers.json
- synonyms: False
- depth: 20  - degree: 13
- answers: 81
- questions: 279
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  7
- answer limit: 0  - maximum loaded:  1


In [12]:
# check that we don't have any answer synonyms
for answer_candidate in human_data_train.answer_synonyms:
    assert len(human_data_train.answer_synonyms[answer_candidate]) == 1

In [15]:
from collections import defaultdict


system = """You are generating semantically similar paraphrases for a given response inside <response> tags to some question inside a <question> tag. 
The generated response paraphrases should be human-like and short, using frequently used words and phrases only.
The generated response paraphrases should also still be a plausible answer to the question.
Output only the generated paraphrases, separating each paraphrase with a <br> tag."""

def user(node_text: str, answer_text: str, num_paraphrases: int) -> str:
    return f"""Generate {num_paraphrases} paraphrases for the response <response>{answer_text}</response> to the question <question>{node_text}</question>."""

NUM_PARAPHRASES = 25
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 10000

set_seed(42)

generated = defaultdict(lambda: set())


for idx, node in tqdm(enumerate(human_data_train.nodes_by_type[NodeType.QUESTION])):
    for answer in node.answers:
        prompt = generate_prompt(system=system, user=user(node.text, answer.text, NUM_PARAPHRASES))
        answers = generate_output(prompt=prompt, temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
        generated[answer.key] = generated[answer.key].union(answers)
    

0it [00:00, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


 - duplicates: 0


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


 - duplicates: 0


1it [00:57, 57.77s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

 - duplicates: 0


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


 - duplicates: 0


22it [15:55, 43.41s/it]

 - duplicates: 0


In [16]:
import json
with open("../../../resources/en/reimburse/generated/llama3/train_answers_v2.json", "w") as f:
    formatted = {}
    for answer_key in generated:
        formatted[answer_key] = list(generated[answer_key])
    json.dump(formatted, f)
